In [9]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import scipy.io as sio
import liana as li
from liana.method import cellphonedb, geometric_mean

# single source of truth for the permutation-test seed used everywhere below —
# change this one value if you ever want a different (still reproducible) draw
RNG_SEED = 1337
np.random.seed(RNG_SEED)   # belt-and-suspenders global seed, on top of passing seed= explicitly to custom_aggregate()

export_dir = "ref_export"

counts = sio.mmread(f"{export_dir}/lognormalized_matrix.mtx").tocsr().T
genes = pd.read_csv(f"{export_dir}/features.tsv", header=None)[0].values
barcodes = pd.read_csv(f"{export_dir}/barcodes.tsv", header=None)[0].values
meta = pd.read_csv(f"{export_dir}/metadata.csv").set_index("barcode").loc[barcodes]

adata = ad.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
adata.raw = adata   # already log-normalized from R — rank_aggregate/AggregateClass default to use_raw=True

# candidate_resource.csv now pools the significant candidate LR pairs from all 4
# spatial pipelines (Astrocyte-Vascular, Vascular-Microglia, OPC-Cajal-Retzius,
# Neurogenic_lineage-GC) — see the R "Save data for LIANA+ CCC" section. Running
# the permutation test once on the pooled resource, then filtering per cell-type
# pair below, avoids paying the (expensive) permutation cost 4 times.
candidate_resource = pd.read_csv(f"{export_dir}/candidate_resource.csv")

# mirror ref_seurat$Celltype_updated from 03_RCTD_SpotDeconvolution.qmd exactly
# (build-full-reference chunk), so LIANA's "Celltype" groups match the RCTD
# cell types used throughout the spatial pipeline:
#   ref_seurat$Celltype_updated <- ref_seurat$Celltype %>%
#     gsub("qNSC|nIPC|Neuroblast","Neurogenic_lineage", .) %>%
#     gsub("SMC|Pericyte|Endothelial","Vascular", .)
celltype_merge_map = {
    "qNSC": "Neurogenic_lineage",
    "nIPC": "Neurogenic_lineage",
    "Neuroblast": "Neurogenic_lineage",
    "SMC": "Vascular",
    "Pericyte": "Vascular",
    "Endothelial": "Vascular",
}
adata.obs["Celltype"] = adata.obs["Celltype"].replace(celltype_merge_map)

adata.obs["Celltype"].value_counts()   # confirm the merge worked, and that Microglia / OPC / Cajal-Retzius / Neurogenic_lineage are present


methods = [cellphonedb, geometric_mean]
custom_aggregate = li.mt.AggregateClass(li.mt.aggregate_meta, methods=methods)

results = []
for age_label in ["Young", "Middle", "Old"]:
    sub = adata[adata.obs["Age"] == age_label].copy()

    custom_aggregate(sub,
                      groupby="Celltype",
                      resource=candidate_resource,
                      expr_prop=0.1,
                      n_perms=1000,
                      seed=RNG_SEED,
                      verbose=True,
                      key_added="cpdb_gmean_agg")

    res_age = sub.uns["cpdb_gmean_agg"].copy()
    res_age["Age"] = age_label
    results.append(res_age)

res_by_age = pd.concat(results, ignore_index=True)


Using `.raw`!
Converting to sparse csr matrix!


Using provided `resource`.


11004 features of mat are empty, they will be removed.
Converting `Celltype` to categorical!
/home/vkorob/miniconda3/envs/liana-env/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:307: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/vkorob/miniconda3/envs/liana-env/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:167: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.01 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 5382 samples and 406 features
Running CellPhoneDB


100%|█████████████████████████████████████| 1000/1000 [00:06<00:00, 153.85it/s]


...

9461 features of mat are empty, they will be removed.
Converting `Celltype` to categorical!
/home/vkorob/miniconda3/envs/liana-env/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:307: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/vkorob/miniconda3/envs/liana-env/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:167: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.00 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 20259 samples and 409 features
Running CellPhoneDB


100%|██████████████████████████████████████| 1000/1000 [00:15<00:00, 62.96it/s]


Running Geometric Mean


100%|██████████████████████████████████████| 1000/1000 [00:16<00:00, 60.20it/s]


In [10]:
# res_by_age already has every cell-type pair present in the data (groupby="Celltype"
# runs CCC across all pairs at once) — extract_pair() just filters the one shared
# result table per pair of interest, both interaction directions included.

def extract_pair(res_df, ct1, ct2):
    return res_df[
        ((res_df["source"] == ct1) & (res_df["target"] == ct2)) |
        ((res_df["source"] == ct2) & (res_df["target"] == ct1))
    ].copy()

def finalize_and_save(res_pair, filename):
    res_pair = res_pair.rename(columns={"Age": "age"})
    res_pair["age"] = res_pair["age"].str.lower()
    res_pair.to_csv(filename, index=False)
    return res_pair

res_astro_vasc_by_age    = finalize_and_save(extract_pair(res_by_age, "Astrocyte", "Vascular"),
                                              "liana_astro_vasc_by_age.csv")
res_vasc_microglia_by_age = finalize_and_save(extract_pair(res_by_age, "Vascular", "Microglia"),
                                               "liana_vasc_microglia_by_age.csv")
res_opc_cr_by_age        = finalize_and_save(extract_pair(res_by_age, "OPC", "Cajal-Retzius"),
                                              "liana_opc_cr_by_age.csv")
res_neuro_gc_by_age      = finalize_and_save(extract_pair(res_by_age, "Neurogenic_lineage", "GC"),
                                              "liana_neuro_gc_by_age.csv")

# sanity check: each should have rows, both interaction directions, all 3 ages
for name, df in [("astro_vasc", res_astro_vasc_by_age),
                  ("vasc_microglia", res_vasc_microglia_by_age),
                  ("opc_cr", res_opc_cr_by_age),
                  ("neuro_gc", res_neuro_gc_by_age)]:
    print(name, "rows:", len(df), "| sources:", df["source"].unique(), "| ages:", df["age"].unique())


astro_vasc rows: 321 | sources: ['Vascular' 'Astrocyte'] | ages: ['young' 'middle' 'old']
vasc_microglia rows: 244 | sources: ['Vascular' 'Microglia'] | ages: ['young' 'middle' 'old']
opc_cr rows: 438 | sources: ['OPC' 'Cajal-Retzius'] | ages: ['young' 'middle' 'old']
neuro_gc rows: 692 | sources: ['Neurogenic_lineage' 'GC'] | ages: ['young' 'middle' 'old']
